## 🧬 4. Fase EXPLORE: Análise Exploratória Multivariada

Cruzamento das *features* consolidadas na ABT (`data/processed/abt_sanidade_vegetal.csv`)[cite: 1] para mapear correlações lineares, identificar variáveis redundantes (multicolinearidade) e descobrir padrões ocultos de separabilidade cruzando as métricas com a classe alvo.

In [2]:
# 4.1 Setup do Motor Multivariado (Plotly Avançado)
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, Markdown

class AdvancedMultivariateEDA:
    """
    Pipeline interativo de Análise Multivariada.
    Focado em matrizes de correlação, multicolinearidade e projeção de separabilidade.
    """
    def __init__(self, df: pd.DataFrame, target_col: str):
        self.df = df
        self.target_col = target_col
        self.num_cols = self.df.select_dtypes(include=['float64', 'int64']).columns.tolist()
        
        # Isolando apenas as features contínuas reais para a matriz (exclui IDs e targets numéricos)
        cols_to_exclude = ['target_binary', 'target_multiclass', 'sample_id']
        self.features = [c for c in self.num_cols if c not in cols_to_exclude]

    def plot_correlation_heatmap(self):
        """Gera um Heatmap interativo da Matriz de Correlação (Pearson)."""
        display(Markdown("### 🧮 Matriz de Correlação Global"))
        
        corr_matrix = self.df[self.features].corr().round(2)
        
        fig = go.Figure(data=go.Heatmap(
            z=corr_matrix.values,
            x=corr_matrix.columns,
            y=corr_matrix.index,
            colorscale='RdBu_r',
            zmin=-1, zmax=1,
            text=corr_matrix.values,
            texttemplate="%{text}",
            hoverinfo="x+y+z"
        ))
        
        fig.update_layout(
            title="Heatmap de Correlação de Features (Pearson)",
            template="plotly_dark",
            height=700,
            width=900,
            xaxis_showgrid=False,
            yaxis_showgrid=False
        )
        fig.show()

    def plot_feature_scatter_cross(self, x_col: str, y_col: str):
        """Cruza variáveis específicas com a classe alvo em um gráfico de dispersão."""
        display(Markdown(f"### 🌌 Cruzamento Estratégico: `{x_col}` vs `{y_col}`"))
        
        fig = px.scatter(
            self.df, x=x_col, y=y_col, color=self.target_col,
            color_discrete_sequence=px.colors.qualitative.Bold,
            title=f"Projeção de Separabilidade Multivariada: {x_col} vs {y_col}",
            template="plotly_dark",
            opacity=0.75,
            hover_data=[self.target_col],
            render_mode='svg' # Parâmetro adicionado para contornar o bloqueio de WebGL
        )
        
        fig.update_traces(marker=dict(size=7, line=dict(width=0.5, color='DarkSlateGrey')))
        fig.update_layout(height=600)
        fig.show()

# 4.2 Execução do Pipeline Multivariado
abt_path = '../data/processed/abt_sanidade_vegetal.csv'
df_features = pd.read_csv(abt_path)

multi_eda = AdvancedMultivariateEDA(df=df_features, target_col='class_label')

# 1. Gera a matriz para analisar redundâncias
multi_eda.plot_correlation_heatmap()

# 2. Gráfico de dispersão cruzando as melhores features (Textura GLCM vs Cor ExG)
# Escolhemos essas duas com base no contraste entre saúde estrutural e necrose
multi_eda.plot_feature_scatter_cross(x_col='exg_index', y_col='glcm_contrast')

### 🧮 Matriz de Correlação Global

### 🌌 Cruzamento Estratégico: `exg_index` vs `glcm_contrast`

## 🧩 Conclusões da Análise Multivariada e Próximos Passos

✅ **Padrões Ocultos e Hipóteses Registradas (Checklist):**

1. **Variáveis Redundantes (Multicolinearidade):** A matriz de correlação revelou redundância extrema (correlação > 0.95) entre os metadados de dimensão e peso do arquivo (`width`, `height` e `size_kb`). 
   * *Hipótese Estratégica:* Precisamos **remover (dropar)** essas variáveis numéricas durante a modelagem. O tamanho da imagem não é um preditor real de saúde vegetal e causará ruído (overfitting) no classificador.
2. **Identificação de Padrões Ocultos:** Observa-se que a Homogeneidade (`glcm_homogeneity`) possui forte correlação negativa (-0.46) com a doença, enquanto o Contraste (`glcm_contrast`) aumenta nas folhas doentes. Isso indica que fungos e necroses "quebram" a textura lisa da folha.
3. **Cruzamento com a Classe Alvo:** O gráfico de dispersão demonstra que o cruzamento de índices espectrais (como o Excesso de Verde - `exg_index`) com métricas de textura cria agrupamentos (*clusters*) visualmente separáveis. Isso valida a viabilidade de aplicar algoritmos como Support Vector Machines (SVM) na próxima Sprint.